**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Beyond Kalman: EKF, UKF & Particle Filters

The [Kalman filter](./Intro_AdFilt_KF.ipynb) is optimal for linear dynamics and Gaussian noise. Reality is neither. Three sessions on the escalation ladder: linearize (EKF), sample deterministically (UKF), sample massively (particle filter) — each demonstrated on problems where its predecessor fails.

## 1. Pre-requisites

[Adaptive Filtering: Kalman](./Intro_AdFilt_KF.ipynb) — this workshop assumes its notation and predict/update instincts.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2)

---
### 🕐 Session 1 of 3 — *The Extended Kalman Filter* (~35 min)
**Goal:** linearize the model at the current estimate; track a pendulum.
**Builds on:** [Kalman workshop](./Intro_AdFilt_KF.ipynb). &nbsp; **Feeds into:** Session 2 (UKF).

---

## 2. EKF: Pretend It's Linear (Locally)

💡 **Intuition.** Nonlinear dynamics $f(\mathbf{x})$ break the Kalman derivation. EKF's fix is a first-order Taylor patch: propagate the *mean* through the true $f$, but propagate the *covariance* through $f$'s Jacobian at the current estimate — a fresh linearization every step. It works beautifully while the uncertainty stays small enough that $f$ is locally straight; it lies when curvature bites within one standard deviation.

In [2]:
# Pendulum: x = [angle, angular velocity], observe only sin(angle) (e.g. a horizontal position sensor)
dt, g_l = 0.02, 9.81
def f(x):  return np.array([x[0] + dt*x[1], x[1] - dt*g_l*np.sin(x[0])])
def F_jac(x): return np.array([[1, dt], [-dt*g_l*np.cos(x[0]), 1]])
def h(x):  return np.array([np.sin(x[0])])
def H_jac(x): return np.array([[np.cos(x[0]), 0.0]])

T = 600
Q = np.diag([1e-6, 1e-4]); R_m = np.array([[0.02**2]])
xs = np.zeros((T, 2)); xs[0] = [2.2, 0]                     # LARGE swing: genuinely nonlinear
for t in range(1, T):
    xs[t] = f(xs[t-1]) + rng.multivariate_normal([0,0], Q)
zs = np.array([h(x) + rng.normal(0, 0.02, 1) for x in xs])

x_e, P = np.array([1.5, 0.5]), np.eye(2)                    # wrong initial guess
est = []
for t in range(T):
    Fj = F_jac(x_e)
    x_e = f(x_e); P = Fj @ P @ Fj.T + Q                     # predict through TRUE f, JACOBIAN for P
    Hj = H_jac(x_e)
    S = Hj @ P @ Hj.T + R_m
    K = P @ Hj.T @ np.linalg.inv(S)
    x_e = x_e + (K @ (zs[t] - h(x_e))).ravel()
    P = (np.eye(2) - K @ Hj) @ P
    est.append(x_e.copy())
est = np.array(est)

plt.figure(figsize=(9, 2.8))
plt.plot(xs[:, 0], "k--", linewidth=1, label="true angle")
plt.plot(est[:, 0], label="EKF estimate")
plt.plot(np.arcsin(np.clip(zs, -1, 1)), ".", markersize=2, alpha=0.3, label="naive arcsin(z)")
plt.legend(); plt.title("EKF tracks a large-swing pendulum from sin(θ) alone")
plt.tight_layout(); plt.show()
print(f"EKF angle RMSE: {np.sqrt(np.mean((est[100:,0]-xs[100:,0])**2)):.4f} rad")

EKF angle RMSE: 0.0082 rad


/tmp/ipykernel_2045392/3642640616.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 3 — *The Unscented Kalman Filter* (~35 min)
**Goal:** replace Jacobians with sigma points; win when curvature bites.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (particle filters).

---

## 3. UKF: Sample the Belief, Not the Slope

💡 **Intuition.** EKF pushes *one* point and a slope through $f$. The unscented transform pushes a handful of **sigma points** — deterministically placed at the mean ± scaled covariance directions — through the *true* nonlinear $f$, then refits mean and covariance to where they landed. No Jacobians (great when $f$ is ugly or black-box), and accurate to second order instead of first. Motto: *it's easier to approximate a distribution than a nonlinear function.*

In [3]:
# The classic curvature demo: push a Gaussian through polar→Cartesian
def polar_to_xy(p): return np.array([p[0]*np.cos(p[1]), p[0]*np.sin(p[1])])
mean_p = np.array([1.0, np.pi/2]); cov_p = np.diag([0.02**2, 0.35**2])   # tight range, WIDE angle

# Monte Carlo truth
samples = rng.multivariate_normal(mean_p, cov_p, 4000)
xy = np.array([polar_to_xy(s) for s in samples])
mc_mean = xy.mean(0)

# EKF-style: linearize at the mean
J = np.array([[np.cos(mean_p[1]), -mean_p[0]*np.sin(mean_p[1])],
              [np.sin(mean_p[1]),  mean_p[0]*np.cos(mean_p[1])]])
ekf_mean = polar_to_xy(mean_p)

# Unscented transform
n_dim, kappa = 2, 1.0
L = np.linalg.cholesky((n_dim + kappa) * cov_p)
sigmas = [mean_p] + [mean_p + L[:, i] for i in range(2)] + [mean_p - L[:, i] for i in range(2)]
Wts = np.array([kappa/(n_dim+kappa)] + [1/(2*(n_dim+kappa))]*4)
mapped = np.array([polar_to_xy(s) for s in sigmas])
ut_mean = Wts @ mapped

plt.figure(figsize=(5, 4))
plt.scatter(xy[:, 0], xy[:, 1], s=2, alpha=0.15, label="truth (Monte Carlo)")
plt.plot(*mc_mean, "k*", markersize=14, label=f"true mean")
plt.plot(*ekf_mean, "rs", markersize=9, label="EKF (linearized): biased outward")
plt.plot(*ut_mean, "g^", markersize=9, label="unscented: nails it")
plt.legend(fontsize=8); plt.axis("equal"); plt.title("Banana problem: curvature defeats the tangent line")
plt.tight_layout(); plt.show()
print(f"mean estimates — MC {mc_mean.round(3)}, EKF {ekf_mean.round(3)}, UT {ut_mean.round(3)}")

mean estimates — MC [-0.005  0.94 ], EKF [0. 1.], UT [0.    0.941]


/tmp/ipykernel_2045392/3712700104.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 3 — *Particle Filters* (~40 min)
**Goal:** represent ANY belief with weighted samples; track through multimodality.
**Builds on:** Session 2.

---

## 4. When the Belief Isn't a Blob

💡 **Intuition.** EKF/UKF still summarize belief as mean + covariance — one Gaussian blob. Some problems are **multimodal**: a robot that observes only its *distance* to a wall genuinely can't distinguish left from right, and the honest belief is two blobs. The particle filter drops the Gaussian religion: carry $N$ weighted samples (*particles*), move each through the dynamics (with noise), reweight by measurement likelihood, and **resample** to cull the walking dead. It's the [LLN](../../Intro_Math/Analysis/Independence.ipynb) as a filter — any shape of belief, at Monte Carlo prices.

In [4]:
# 1-D corridor localization: robot observes |position| (symmetric!) plus noise
T2, Np = 60, 3000
x_true_pos = -2.0                                        # starts LEFT of center
traj, meas = [], []
pos = x_true_pos
for t in range(T2):
    pos += 0.05 + rng.normal(0, 0.02)                    # drifts rightward
    traj.append(pos); meas.append(abs(pos) + rng.normal(0, 0.1))

particles = rng.uniform(-4, 4, Np)                       # know nothing
weights = np.ones(Np)/Np
snapshots = {}
for t in range(T2):
    particles += 0.05 + rng.normal(0, 0.05, Np)          # predict
    lik = np.exp(-(meas[t] - np.abs(particles))**2 / (2*0.1**2))
    weights = lik + 1e-300; weights /= weights.sum()     # update
    if 1/np.sum(weights**2) < Np/2:                      # resample when degenerate
        particles = particles[rng.choice(Np, Np, p=weights)]
        weights = np.ones(Np)/Np
    if t in (2, 20, 55): snapshots[t] = particles.copy()

fig, axes = plt.subplots(1, 3, figsize=(10, 2.6), sharey=True)
for ax, (t, p) in zip(axes, snapshots.items()):
    ax.hist(p, bins=80, range=(-4, 4), density=True)
    ax.axvline(traj[t], color="r", linestyle=":", label="truth")
    ax.set_title(f"t={t}: {'two hypotheses!' if t < 30 else 'disambiguated'}")
    ax.legend(fontsize=7)
plt.suptitle("|x| measurements: belief is honestly bimodal until motion breaks the symmetry", y=1.04)
plt.tight_layout(); plt.show()

/tmp/ipykernel_2045392/4274007459.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


The disambiguation: both hypothesis clusters drift right, but only one keeps matching the measurements as the true robot crosses regions where $|x|$ evolves differently — the impostor cluster starves and dies at resampling. **No Gaussian filter can even represent the question.**

Costs to respect: $O(N_p)$ per step, particle death in high dimensions (the curse), and resampling noise. The escalation rule: EKF if mildly nonlinear, UKF if curvy or Jacobian-hostile, particles if multimodal or seriously non-Gaussian — never more machinery than the problem demands.

## 5. Conclusion

Linearize, sigma-sample, or particle-sample: three ways to keep the predict/update heartbeat when the world stops being linear. You now own the full state-estimation ladder from LMS to Monte Carlo.

---
## Where next

- [Recurrent Neural Networks](./Intro_RNN.ipynb) — *learn* the dynamics instead of modeling them.
- [Uncertainty in ML](../Intro_Mach_Learn/Uncertainty_in_ML.ipynb) — ensembles: particle filtering's spirit in deep learning.